In [42]:
# Standard libraries
import os
from pathlib import Path
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd


In [43]:
df_80 = pd.read_csv('un_lolazo_submission.csv')
df_81 = pd.read_csv('rf_por_region.csv')
df_85 = pd.read_csv('cat_boost_perh.csv')

In [44]:
n = len(df_80)

In [45]:
df_80_81 = pd.merge(df_80,df_81,on='SamplingOperations_code')

In [46]:
df_80_81['IBD_EQR_Status_x'].rename

<bound method Series.rename of 0       Moderate
1           Good
2       Moderate
3           Good
4           Good
          ...   
5058    Moderate
5059    Moderate
5060        High
5061        High
5062        High
Name: IBD_EQR_Status_x, Length: 5063, dtype: object>

In [47]:
df_80

,SamplingOperations_code,IBD_EQR_Status
0,S02000010_20080811,Moderate
1,S02000010_20100719,Good
2,S02000010_20150811,Moderate
3,S02000010_20170703,Good
4,S02000011_20100719,Good
...,...,...
5058,S06940940_20100708,Moderate
5059,S06940940_20230623,Moderate
5060,S06960950_20160629,High
5061,S06960950_20180719,High


In [48]:
import pandas as pd

# Cargar las predicciones
a = df_80
b = df_81
c = df_85
 
# Unir por id
df = a.merge(b, on="SamplingOperations_code", suffixes=("_81", "_80"))
df = df.merge(c, on="SamplingOperations_code")
df.rename(columns={"IBD_EQR_Status": "IBD_EQR_Status_85"}, inplace=True)


In [49]:

# Votación mayoritaria
df["consenso"] = df[["IBD_EQR_Status_81", "IBD_EQR_Status_80", "IBD_EQR_Status_85"]].mode(axis=1)[0]

# Coincidencia total (los 3 iguales)
df["coinciden_todos"] = (
    (df["IBD_EQR_Status_81"] == df["IBD_EQR_Status_80"]) & 
    (df["IBD_EQR_Status_81"] == df["IBD_EQR_Status_85"])
)

# Proporción de coincidencia
p_coincidencia = df["coinciden_todos"].mean()

print(f"Coincidencia total entre los 3 modelos: {p_coincidencia:.2%}")

# Guardar la predicción final (de consenso)
pred_coincidecnias = df[["SamplingOperations_code", "consenso"]]

Coincidencia total entre los 3 modelos: 79.30%


In [50]:
pred_coincidecnias

,SamplingOperations_code,consenso
0,S02000010_20080811,Good
1,S02000010_20100719,Good
2,S02000010_20150811,Moderate
3,S02000010_20170703,Good
4,S02000011_20100719,Good
...,...,...
5058,S06940940_20100708,Moderate
5059,S06940940_20230623,Moderate
5060,S06960950_20160629,High
5061,S06960950_20180719,High


In [51]:
# Nombres de columnas (ajústalos si cambian)
col_id = "SamplingOperations_code"
col_a  = "IBD_EQR_Status_81"  # Modelo A (81%)
col_b  = "IBD_EQR_Status_80"  # Modelo B (80%)
col_c  = "IBD_EQR_Status_85"  # Modelo C (85%)

# =========================
# 1) Coincidencias par a par
# =========================
df["coincide_ab"] = df[col_a] == df[col_b]
df["coincide_ac"] = df[col_a] == df[col_c]
df["coincide_bc"] = df[col_b] == df[col_c]

# DFs filtrados SOLO con coincidencias (si prefieres ver solo los que sí coinciden)
df_ab = df.loc[df["coincide_ab"], [col_id, col_a, col_b]].copy()
df_ac = df.loc[df["coincide_ac"], [col_id, col_a, col_c]].copy()
df_bc = df.loc[df["coincide_bc"], [col_id, col_b, col_c]].copy()

# (opcional) también puedes quedarte con el flag en versión "completa":
# df_ab_full = df[[col_id, col_a, col_b, "coincide_ab"]].copy()
# df_ac_full = df[[col_id, col_a, col_c, "coincide_ac"]].copy()
# df_bc_full = df[[col_id, col_b, col_c, "coincide_bc"]].copy()

# =========================
# 2) Triple coincidencia
# =========================
df["coinciden_todos"] = (df[col_a] == df[col_b]) & (df[col_a] == df[col_c])
df_triple = df.loc[df["coinciden_todos"], [col_id, col_a, col_b, col_c]].copy()

# (métrica) proporción de triple coincidencia
p_triple = df["coinciden_todos"].mean()
print(f"Triple coincidencia (A=B=C): {p_triple:.2%}")

# =========================
# 3) Votación mayoritaria + % de coincidencia con A, B y C
# =========================
# Predicción de consenso por mayoría
df["consenso"] = df[[col_a, col_b, col_c]].mode(axis=1)[0]

# % de acuerdo del consenso con cada modelo
p_consenso_con_a = (df["consenso"] == df[col_a]).mean()
p_consenso_con_b = (df["consenso"] == df[col_b]).mean()
p_consenso_con_c = (df["consenso"] == df[col_c]).mean()

print(f"Consenso vs A (81%): {p_consenso_con_a:.2%}")
print(f"Consenso vs B (80%): {p_consenso_con_b:.2%}")
print(f"Consenso vs C (85%): {p_consenso_con_c:.2%}")

# DF final de consenso (id + predicción)
df_consenso = df[[col_id, "consenso"]].copy()

# =========================
# (Opcional) Guardados
# =========================
# df_ab.to_csv("results/coincidencias_ab.csv", index=False)
# df_ac.to_csv("results/coincidencias_ac.csv", index=False)
# df_bc.to_csv("results/coincidencias_bc.csv", index=False)
# df_triple.to_csv("results/triple_coincidencia.csv", index=False)
# df_consenso.to_csv("results/prediccion_consenso.csv", index=False)


Triple coincidencia (A=B=C): 79.30%
Consenso vs A (81%): 91.37%
Consenso vs B (80%): 94.71%
Consenso vs C (85%): 93.07%


In [52]:
print(len(df_ab)/n)
print(len(df_bc)/n)
print(len(df_ac)/n)

0.8619395615247877
0.8779379814339324
0.8445585621173217
